In [1]:
!pip install xgboost lightgbm catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.2 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import time
import warnings
warnings.filterwarnings('ignore')


In [4]:
columns_names =['id', 'name', 'gender', 'age', 'city', 'occupation_status', 'profession',
                'academic_pressure', 'work_pressure', 'cgpa', 'study_satisfaction', 'job_satisfaction',
                'sleep_duration', 'dietary_habits', 'degree', 'suicidal_thoughts', 'work_study_hours',
                'financial_stress', 'family_history_mental_illness', 'depression']

In [5]:
columns_test_names =['id', 'name', 'gender', 'age', 'city', 'occupation_status', 'profession',
                     'academic_pressure', 'work_pressure', 'cgpa', 'study_satisfaction', 'job_satisfaction',
                     'sleep_duration', 'dietary_habits', 'degree', 'suicidal_thoughts', 'work_study_hours',
                     'financial_stress', 'family_history_mental_illness']

In [2]:
from google.colab import files

uploaded = files.upload()


Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


In [7]:
train_df = pd.read_csv('train.csv',names=columns_names, header=0)
test_df = pd.read_csv('test.csv',names=columns_test_names, header=0)
submission = pd.read_csv('sample_submission.csv')

In [8]:
train_df

,id,name,gender,age,city,occupation_status,profession,academic_pressure,work_pressure,cgpa,study_satisfaction,job_satisfaction,sleep_duration,dietary_habits,degree,suicidal_thoughts,work_study_hours,financial_stress,family_history_mental_illness,depression
0,0,Aaradhya,Female,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No,0
1,1,Vivan,Male,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No,1
2,2,Yuvraj,Male,33.0,Visakhapatnam,Student,NaN,5.0,NaN,8.97,2.0,NaN,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
3,3,Yuvraj,Male,22.0,Mumbai,Working Professional,Teacher,NaN,5.0,NaN,NaN,1.0,Less than 5 hours,Moderate,BBA,Yes,10.0,1.0,Yes,1
4,4,Rhea,Female,30.0,Kanpur,Working Professional,Business Analyst,NaN,1.0,NaN,NaN,1.0,5-6 hours,Unhealthy,BBA,Yes,9.0,4.0,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140695,140695,Vidya,Female,18.0,Ahmedabad,Working Professional,NaN,NaN,5.0,NaN,NaN,4.0,5-6 hours,Unhealthy,Class 12,No,2.0,4.0,Yes,1
140696,140696,Lata,Female,41.0,Hyderabad,Working Professional,Content Writer,NaN,5.0,NaN,NaN,4.0,7-8 hours,Moderate,B.Tech,Yes,6.0,5.0,Yes,0
140697,140697,Aanchal,Female,24.0,Kolkata,Working Professional,Marketing Manager,NaN,3.0,NaN,NaN,1.0,More than 8 hours,Moderate,B.Com,No,4.0,4.0,No,0
140698,140698,Prachi,Female,49.0,Srinagar,Working Professional,Plumber,NaN,5.0,NaN,NaN,2.0,5-6 hours,Moderate,ME,Yes,10.0,1.0,No,0


In [9]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140700 entries, 0 to 140699
Data columns (total 20 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   id                             140700 non-null  int64  
 1   name                           140700 non-null  object 
 2   gender                         140700 non-null  object 
 3   age                            140700 non-null  float64
 4   city                           140700 non-null  object 
 5   occupation_status              140700 non-null  object 
 6   profession                     104070 non-null  object 
 7   academic_pressure              27897 non-null   float64
 8   work_pressure                  112782 non-null  float64
 9   cgpa                           27898 non-null   float64
 10  study_satisfaction             27897 non-null   float64
 11  job_satisfaction               112790 non-null  float64
 12  sleep_duration                

In [10]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93800 entries, 0 to 93799
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id                             93800 non-null  int64  
 1   name                           93800 non-null  object 
 2   gender                         93800 non-null  object 
 3   age                            93800 non-null  float64
 4   city                           93800 non-null  object 
 5   occupation_status              93800 non-null  object 
 6   profession                     69168 non-null  object 
 7   academic_pressure              18767 non-null  float64
 8   work_pressure                  75022 non-null  float64
 9   cgpa                           18766 non-null  float64
 10  study_satisfaction             18767 non-null  float64
 11  job_satisfaction               75026 non-null  float64
 12  sleep_duration                 93800 non-null 

In [11]:
train_df['degree'].value_counts()

,count
degree,
Class 12,14729
B.Ed,11691
B.Arch,8742
B.Com,8113
B.Pharm,5856
...,...
LCA,1
B B.Com,1
RCA,1


In [12]:
train_df.describe()

,id,age,academic_pressure,work_pressure,cgpa,study_satisfaction,job_satisfaction,work_study_hours,financial_stress,depression
count,140700.000000,140700.000000,27897.000000,112782.000000,27898.000000,27897.000000,112790.000000,140700.000000,140696.000000,140700.000000
mean,70349.500000,40.388621,3.142273,2.998998,7.658636,2.944940,2.974404,6.252679,2.988983,0.181713
std,40616.735775,12.384099,1.380457,1.405771,1.464466,1.360197,1.416078,3.853615,1.413633,0.385609
min,0.000000,18.000000,1.000000,1.000000,5.030000,1.000000,1.000000,0.000000,1.000000,0.000000
25%,35174.750000,29.000000,2.000000,2.000000,6.290000,2.000000,2.000000,3.000000,2.000000,0.000000
50%,70349.500000,42.000000,3.000000,3.000000,7.770000,3.000000,3.000000,6.000000,3.000000,0.000000
75%,105524.250000,51.000000,4.000000,4.000000,8.920000,4.000000,4.000000,10.000000,4.000000,0.000000
max,140699.000000,60.000000,5.000000,5.000000,10.000000,5.000000,5.000000,12.000000,5.000000,1.000000


In [17]:
corr_with_target = train_df.corr(numeric_only=True)['depression'] \
                           .sort_values(ascending=False)

In [18]:
corr_with_target

,depression
depression,1.000000
academic_pressure,0.475037
financial_stress,0.227237
work_pressure,0.216634
work_study_hours,0.191746
cgpa,0.021729
id,0.003944
study_satisfaction,-0.168014
job_satisfaction,-0.168543
age,-0.564671


In [ ]:
#Data Processing

In [20]:
train_df.isnull().sum()

,0
id,0
name,0
gender,0
age,0
city,0
occupation_status,0
profession,36630
academic_pressure,112803
work_pressure,27918
cgpa,112802


In [21]:
test_df.isnull().sum()

,0
id,0
name,0
gender,0
age,0
city,0
occupation_status,0
profession,24632
academic_pressure,75033
work_pressure,18778
cgpa,75034


In [22]:
cols_med=['academic_pressure','work_pressure','cgpa','study_satisfaction','job_satisfaction','financial_stress']

In [23]:
median_values=train_df[cols_med].median()
train_df[cols_med]=train_df[cols_med].fillna(median_values)
test_df[cols_med]=test_df[cols_med].fillna(median_values)

In [24]:
median_values

,0
academic_pressure,3.00
work_pressure,3.00
cgpa,7.77
study_satisfaction,3.00
job_satisfaction,3.00
financial_stress,3.00


In [25]:
train_df.isnull().sum()

,0
id,0
name,0
gender,0
age,0
city,0
occupation_status,0
profession,36630
academic_pressure,0
work_pressure,0
cgpa,0


In [26]:
train_df['profession']=train_df['profession'].fillna('Other')
test_df['profession']=test_df['profession'].fillna('Other')

In [29]:
train_df.isna().sum()

,0
id,0
name,0
gender,0
age,0
city,0
occupation_status,0
profession,0
academic_pressure,0
work_pressure,0
cgpa,0


In [30]:
test_df.isna().sum()

,0
id,0
name,0
gender,0
age,0
city,0
occupation_status,0
profession,0
academic_pressure,0
work_pressure,0
cgpa,0


In [31]:
#cleaning the dietary habits column
valid_diet={
    'Healthy':'Healthy',
    'More Healthy':'Healthy',
    'No Healthy':'Unhealthy',
    'Less than Healthy':'Unhealthy',
    'Less Healthy':'Unhealthy',
    'Unhealthy':'Unhealthy',
    'Moderate':'Moderate'
}

def cleandiet(val):
    if pd.isna(val):
        return 'Other'
    val=str(val).strip()
    return valid_diet.get(val,'Other')

train_df['dietary_habits'] = train_df['dietary_habits'].apply(cleandiet)
test_df['dietary_habits'] = test_df['dietary_habits'].apply(cleandiet)


In [32]:
degree_mode = train_df['degree'].mode()[0]
train_df['degree']=train_df['degree'].fillna(degree_mode)
test_df['degree']=test_df['degree'].fillna(degree_mode)

In [33]:
test_df.isnull().sum()

,0
id,0
name,0
gender,0
age,0
city,0
occupation_status,0
profession,0
academic_pressure,0
work_pressure,0
cgpa,0


In [35]:
train_df['sleep_duration'].unique()

array(['More than 8 hours', 'Less than 5 hours', '5-6 hours', '7-8 hours',
       'Sleep_Duration', '1-2 hours', '6-8 hours', '4-6 hours',
       '6-7 hours', '10-11 hours', '8-9 hours', '40-45 hours',
       '9-11 hours', '2-3 hours', '3-4 hours', 'Moderate', '55-66 hours',
       '4-5 hours', '9-6 hours', '1-3 hours', 'Indore', '45', '1-6 hours',
       '35-36 hours', '8 hours', 'No', '10-6 hours', 'than 5 hours',
       '49 hours', 'Unhealthy', 'Work_Study_Hours', '3-6 hours',
       '45-48 hours', '9-5', 'Pune', '9-5 hours'], dtype=object)

In [36]:
sleep_train= {'More than 8 hours':9, 'Less than 5 hours':4, '5-6 hours':5.5, '7-8 hours':7.5,'1-2 hours':1.5,
              '6-8 hours':7, '4-6 hours':5,'6-7 hours':6.5, '10-11 hours':10.5, '8-9 hours':8, '40-45 hours':4.5,
              '9-11 hours':10, '2-3 hours':2.5, '3-4 hours':3.5, '55-66 hours':5.5,'4-5 hours':4.5, '9-6 hours':9,
              '1-3 hours':2, '45':4.5, '1-6 hours':3,'35-36 hours':5.5, '8 hours':8, 'No':0, '10-6 hours':8,
              'than 5 hours':4.5,'49 hours':7, '3-6 hours':4.5,'45-48 hours':5, '9-5':7, '9-5 hours':8}

In [37]:
train_df['sleep_duration']=train_df['sleep_duration'].map(sleep_train)

In [43]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140700 entries, 0 to 140699
Data columns (total 20 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   id                             140700 non-null  int64  
 1   name                           140700 non-null  object 
 2   gender                         140700 non-null  object 
 3   age                            140700 non-null  float64
 4   city                           140700 non-null  object 
 5   occupation_status              140700 non-null  object 
 6   profession                     140700 non-null  object 
 7   academic_pressure              140700 non-null  float64
 8   work_pressure                  140700 non-null  float64
 9   cgpa                           140700 non-null  float64
 10  study_satisfaction             140700 non-null  float64
 11  job_satisfaction               140700 non-null  float64
 12  sleep_duration                

In [41]:
sleep_median= train_df['sleep_duration'].median()

In [42]:
train_df['sleep_duration']=train_df['sleep_duration'].fillna(sleep_median)

In [44]:
test_df['sleep_duration'].unique()

array(['Less than 5 hours', '7-8 hours', 'More than 8 hours', '5-6 hours',
       '0', 'Meerut', '9-5 hours', '6-7 hours', '60-65 hours', 'Vivan',
       '3-4 hours', '1-6 hours', '9-5', 'Unhealthy', '8-9 hours',
       '4-5 hours', 'than 5 hours', '9-6 hours', '1-2 hours',
       '8-89 hours', 'Have_you_ever_had_suicidal_thoughts', '20-21 hours',
       '10-6 hours', '1-3 hours', '6 hours', '50-75 hours', '4-6 hours',
       '2-3 hours', '9-11 hours', '9-10 hours', '3-6 hours'], dtype=object)

In [45]:
sleep_test= {'Less than 5 hours':4, '7-8 hours':7.5, 'More than 8 hours':9, '5-6 hours':5.5,'0':0, '9-5 hours':8,
             '6-7 hours':6.5, '60-65 hours':6.5, '3-4 hours':3.5, '1-6 hours':3, '9-5':8, '8-9 hours':8.5,
             '4-5 hours':4.5, 'than 5 hours':4.5, '9-6 hours':9, '1-2 hours':1.5,'8-89 hours':8.5, '20-21 hours':7,
             '10-6 hours':8, '1-3 hours':2, '6 hours':6, '50-75 hours':6, '4-6 hours':5,'2-3 hours':2.5,
             '9-11 hours':10, '9-10 hours':9.5, '3-6 hours':4.5}

In [47]:
test_df['sleep_duration']=test_df['sleep_duration'].map(sleep_test)

In [48]:
sleep_median_test=test_df['sleep_duration'].median()

In [49]:
test_df['sleep_duration']=test_df['sleep_duration'].fillna(sleep_median_test)

In [50]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93800 entries, 0 to 93799
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id                             93800 non-null  int64  
 1   name                           93800 non-null  object 
 2   gender                         93800 non-null  object 
 3   age                            93800 non-null  float64
 4   city                           93800 non-null  object 
 5   occupation_status              93800 non-null  object 
 6   profession                     93800 non-null  object 
 7   academic_pressure              93800 non-null  float64
 8   work_pressure                  93800 non-null  float64
 9   cgpa                           93800 non-null  float64
 10  study_satisfaction             93800 non-null  float64
 11  job_satisfaction               93800 non-null  float64
 12  sleep_duration                 93800 non-null 

In [66]:
train_df['degree'].value_counts()

,count
degree,
Class 12,14731
B.Ed,11691
B.Arch,8742
B.Com,8113
B.Pharm,5856
...,...
LCA,1
B B.Com,1
RCA,1


In [67]:
train_len=len(train_df)



combined = pd.concat([train_df, test_df], axis=0)
combined = combined.drop(columns=['id', 'name'], errors='ignore')


cat_cols=['gender','city','occupation_status','profession','dietary_habits','degree','suicidal_thoughts','family_history_mental_illness']

top_prof = train_df['profession'].value_counts().nlargest(10).index
combined['profession'] = combined['profession'].where(combined['profession'].isin(top_prof), 'Other')
top_city = train_df['city'].value_counts().nlargest(10).index
combined['city'] = combined['city'].where(combined['city'].isin(top_city), 'Other')
top_degree = train_df['degree'].value_counts().nlargest(10).index
combined['degree'] = combined['degree'].where(combined['degree'].isin(top_degree), 'Other')

combined_encoded = pd.get_dummies(combined, columns=cat_cols, drop_first=True)

# Split back into train and test
train_df_encoded = combined_encoded.iloc[:train_len, :]
test_df_encoded = combined_encoded.iloc[train_len:, :]

test_df_encoded = test_df_encoded.drop(columns=['depression'], errors='ignore')

In [69]:
test_df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Index: 93800 entries, 0 to 93799
Data columns (total 45 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   age                                     93800 non-null  float64
 1   academic_pressure                       93800 non-null  float64
 2   work_pressure                           93800 non-null  float64
 3   cgpa                                    93800 non-null  float64
 4   study_satisfaction                      93800 non-null  float64
 5   job_satisfaction                        93800 non-null  float64
 6   sleep_duration                          93800 non-null  float64
 7   work_study_hours                        93800 non-null  float64
 8   financial_stress                        93800 non-null  float64
 9   gender_Male                             93800 non-null  bool   
 10  city_Kalyan                             93800 non-null  bool   

In [74]:
#feature importance
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Features and target
X = train_df_encoded.drop(columns=['depression'])
y = train_df['depression']

# Train a RandomForest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Get feature importance
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.head(20))


age                                       0.296770
suicidal_thoughts_Yes                     0.137327
academic_pressure                         0.136737
occupation_status_Working Professional    0.093004
cgpa                                      0.058745
study_satisfaction                        0.051868
financial_stress                          0.047322
degree_Class 12                           0.039094
work_pressure                             0.035843
job_satisfaction                          0.025443
work_study_hours                          0.021794
profession_Other                          0.019544
dietary_habits_Unhealthy                  0.012599
sleep_duration                            0.004932
profession_Teacher                        0.003035
degree_Other                              0.002641
profession_Content Writer                 0.001329
dietary_habits_Moderate                   0.001269
family_history_mental_illness_Yes         0.001034
gender_Male                    

Modelling

In [75]:
models = {"XGBoost": xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,random_state=42,
                                       n_jobs=-1, eval_metric='logloss', verbosity=0),
          "LightGBM": lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42,
                                         n_jobs=-1, verbose=-1),
          "CatBoost": CatBoostClassifier(n_estimators=500, learning_rate=0.05, depth=6, random_state=42,
                                         verbose=0, allow_writing_files=False),
          "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10,random_state=42, n_jobs=-1)}

results = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    start_time = time.time()
    cv_results = cross_val_score(model, X, y, cv=skf, scoring='accuracy', n_jobs=-1)
    elapsed_time = time.time() - start_time

    results.append({"Model": name,"Mean Accuracy": cv_results.mean(),"Std Dev": cv_results.std(),
                    "Min Score": cv_results.min(),"Max Score": cv_results.max(),"Time (s)": elapsed_time})

    print(f"{name}: {cv_results.mean():.5f}")

results_df = pd.DataFrame(results).sort_values(by="Mean Accuracy", ascending=False)

XGBoost: 0.93824
LightGBM: 0.93848
CatBoost: 0.93928
Random Forest: 0.93252


In [76]:
results_df

,Model,Mean Accuracy,Std Dev,Min Score,Max Score,Time (s)
2,CatBoost,0.939275,0.001161,0.937704,0.941009,90.362794
1,LightGBM,0.938479,0.001425,0.936532,0.940441,96.594613
0,XGBoost,0.938244,0.001149,0.936994,0.939801,44.242797
3,Random Forest,0.932516,0.001506,0.930597,0.935039,96.556088


In [77]:
from sklearn.ensemble import VotingClassifier

clf1 = xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1,
                         eval_metric='logloss', verbosity=0)
clf2 = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
clf3 = CatBoostClassifier(n_estimators=500, learning_rate=0.05, depth=6, random_state=42, verbose=0,
                          allow_writing_files=False)

voting_clf = VotingClassifier(estimators=[('xgb', clf1), ('lgb', clf2), ('cat', clf3)],voting='soft',weights=[1, 1, 2])

voting_clf.fit(X, y)

VotingClassifier(estimators=[('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric='logloss',
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_co...
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=500, n_jobs=-1,
                                            num_parallel_tree=None, ...)),
                             ('lgb',
                              LGBMClassifier(learning_rate=0.05, max_depth=6,
                                             n_estimators=500, n_jobs=-1,
                                             random_state=42, verbose=-1)),
                             ('cat',
                              <catboost.core.CatBoostClassifier object at 0x7cb73ee5a690>)],
                 voting='soft', weights=[1, 1, 2])

In [79]:
test_preds_proba = voting_clf.predict_proba(test_df_encoded)[:, 1]
final_preds = (test_preds_proba > 0.5).astype(int)

In [80]:
final_preds

array([0, 0, 0, ..., 0, 1, 0])

In [81]:
final_preds_df=pd.DataFrame(final_preds,columns=['depression'])

In [85]:
final_preds_df

,depression
0,0
1,0
2,0
3,1
4,0
...,...
93795,0
93796,1
93797,0
93798,1


In [87]:
import joblib
clean_sleep_mapping = {'More than 8 hours':9, 'Less than 5 hours':4, '5-6 hours':5.5, '7-8 hours':7.5,'1-2 hours':1.5,
                 '6-8 hours':7, '4-6 hours':5,'6-7 hours':6.5, '10-11 hours':10.5, '8-9 hours':8,
                 '40-45 hours':4.5,'9-11 hours':10, '2-3 hours':2.5, '3-4 hours':3.5, '55-66 hours':5.5,
                 '4-5 hours':4.5, '9-6 hours':9, '1-3 hours':2, '45':4.5, '1-6 hours':3,'35-36 hours':5.5,
                 '8 hours':8, 'No':0, '10-6 hours':8, 'than 5 hours':4.5,'49 hours':7, '3-6 hours':4.5,
                 '45-48 hours':5, '9-5':7, '9-5 hours':8,'0':0,'9-5 hours':8, '60-65 hours':6.5,'4-5 hours':4.5,
                 '8-89 hours':8.5, '20-21 hours':7,'6 hours':6, '50-75 hours':6, '9-10 hours':9.5}
model_artifact = {
    "model": voting_clf,
    "feature_columns": X.columns.tolist(),
    "sleep_mapping": clean_sleep_mapping
}

joblib.dump(model_artifact, "depression_model.pkl")


['depression_model.pkl']

In [88]:
from google.colab import files

files.download("depression_model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>